In [3]:
import xarray as xr
import os
import sys

import logging

# graphcast is in the parent directory so insert it into the path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import argparse
import dataclasses
import xarray as xr
import numpy as np
import pandas as pd
from datetime import datetime
from tqdm import tqdm

import jax
import optax

from graphcast import checkpoint, data_utils, rollout, graphcast, normalization
import save_params_utils
import setup_jax_functions
from plotting import scale, select, plot_data, save_animation, save_static_plot
from metrics import compute_rmse, compute_mae, compute_bias, compute_acc
import matplotlib.pyplot as plt
# import pynvml
import time

current_date = datetime.now().strftime("%Y-%m-%d_%H-%M")

# jax.config.update('jax_disable_jit', True)

# for memory efficiency
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE']='false'
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '2.0' 
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"]="platform"

mean_by_level = None
stddev_by_level = None
diffs_stddev_by_level = None
model_config = None
task_config = None
params = None
state = None


# modify the gradients function signature (needed for finetuning with optax)
def grads_fn(params, state, inputs, targets, forcings, model_config, task_config):
    def _aux(params, state, i, t, f):
        (loss, diagnostics), next_state = setup_jax_functions.loss_fn.apply(params, state, jax.random.PRNGKey(0), model_config, task_config, i, t, f)
        return loss, (diagnostics, next_state)
    (loss, (diagnostics, next_state)), grads = jax.value_and_grad(_aux, has_aux=True)(params, state, inputs, targets, forcings)
    return loss, diagnostics, next_state, grads



In [4]:
wb_data = xr.open_zarr("/Datastorage/saptarishi.dhanuka_asp25/era5_data/wb_era5_jan2016_temp_ppt.zarr/")
wb_data

<xarray.Dataset> Size: 3GB
Dimensions:                  (time: 124, latitude: 181, longitude: 361,
                              level: 13)
Coordinates:
  * latitude                 (latitude) float64 1kB -90.0 -89.0 ... 89.0 90.0
  * level                    (level) int64 104B 50 100 150 200 ... 850 925 1000
  * longitude                (longitude) float64 3kB 0.0 1.0 2.0 ... 359.0 360.0
  * time                     (time) datetime64[ns] 992B 2016-01-01 ... 2016-0...
Data variables: (12/13)
    10m_u_component_of_wind  (time, latitude, longitude) float32 32MB dask.array<chunksize=(1, 181, 361), meta=np.ndarray>
    10m_v_component_of_wind  (time, latitude, longitude) float32 32MB dask.array<chunksize=(1, 181, 361), meta=np.ndarray>
    2m_temperature           (time, latitude, longitude) float32 32MB dask.array<chunksize=(1, 181, 361), meta=np.ndarray>
    geopotential             (time, level, latitude, longitude) float32 421MB dask.array<chunksize=(1, 13, 181, 361), meta=np.ndarray>
    geopotential_at_surface  (latitude, longitude) float32 261kB dask.array<chunksize=(181, 361), meta=np.ndarray>
    land_sea_mask            (latitude, longitude) float32 261kB dask.array<chunksize=(181, 361), meta=np.ndarray>
    ...                       ...
    specific_humidity        (time, level, latitude, longitude) float32 421MB dask.array<chunksize=(1, 13, 181, 361), meta=np.ndarray>
    temperature              (time, level, latitude, longitude) float32 421MB dask.array<chunksize=(1, 13, 181, 361), meta=np.ndarray>
    total_precipitation_6hr  (time, latitude, longitude) float32 32MB dask.array<chunksize=(1, 181, 361), meta=np.ndarray>
    u_component_of_wind      (time, level, latitude, longitude) float32 421MB dask.array<chunksize=(1, 13, 181, 361), meta=np.ndarray>
    v_component_of_wind      (time, level, latitude, longitude) float32 421MB dask.array<chunksize=(1, 13, 181, 361), meta=np.ndarray>
    vertical_velocity        (time, level, latitude, longitude) float32 421MB dask.array<chunksize=(1, 13, 181, 361), meta=np.ndarray>

In [5]:
global mean_by_level
global stddev_by_level
global diffs_stddev_by_level
global model_config
global task_config
global params
global state

In [6]:
arco = wb_data

filename = '/Datastorage/saptarishi.dhanuka_asp25/gc_weights/origs/graphcast_1_13.npz'

old_lats = arco['latitude'].values
old_lons = arco['longitude'].values
new_lats = np.arange(-90.0, 90.0 + 1e-8, 1.0)
new_lats = np.flip(new_lats)
new_lons = np.arange(0, 359.75 + 1e-8, 1.0)
arco = arco.interp({'latitude': new_lats, 'longitude': new_lons}, 
                        method='linear',
                        kwargs={'fill_value': None})

input_vars = ['10m_u_component_of_wind',
            'geopotential_at_surface',
            '10m_v_component_of_wind',
            'specific_humidity',
            'land_sea_mask',
            'vertical_velocity',
            'geopotential',
            'v_component_of_wind',
            'temperature',
            'total_precipitation_6hr',
            'mean_sea_level_pressure',
            '2m_temperature',
            'u_component_of_wind']

# arco = arco.drop_vars(['toa_incident_solar_radiation',
# 'year_progress_sin',
# 'year_progress_cos',
# 'day_progress_sin',
# 'day_progress_cos','cos_latitude',
# 'cos_longitude','sin_longitude'])

# arco = arco.rename({'total_precipitation': 'total_precipitation_6hr'})

arco = arco.expand_dims(batch=1)
arco = arco.rename({'latitude': 'lat', 'longitude': 'lon'})

datetime_array = arco['time'].values
# Calculate the time coordinate in 6-hour increments (in nanoseconarco)
time_array = np.arange(0, len(datetime_array) * 21600000000000, 21600000000000, dtype='timedelta64[ns]')

# Add the new 'time' coordinate to the dataset
arco1 = arco.assign_coords(datetime=('time', time_array))

temp_time = arco1.coords["time"].copy()
temp_datetime = arco1.coords["datetime"].copy()

# Reassign the coordinates, swapping their values
arco1 = arco1.assign_coords(
    time=temp_datetime,
    datetime=temp_time
)


# arco1['geopotential_at_surface'] = arco1['geopotential_at_surface'].isel(batch=0, time=0)
# arco1['land_sea_mask'] = arco1['land_sea_mask'].isel(batch=0, time=0)

old_datetime = arco1["datetime"].values  # shape (1489,)

# For our purposes, we want the coordinate to have shape (batch, time). Since the batch
# dimension is of length 1, we can simply add a new axis.
new_datetime = old_datetime[np.newaxis, :]  # shape becomes (1, 1489)

# Now, reassign the "datetime" coordinate to have dims ("batch", "time").
arco1 = arco1.assign_coords(datetime=(("batch", "time"), new_datetime))

print(f"Coordinates after reassigning: {arco1.coords}\n")



select_time = arco1.isel(time=slice(0, 240))
tik = datetime.now()

select_time_eval = arco1.isel(time=slice(7,71))

eval_trial_batch = select_time_eval.load()

print(f"Eval batch first time: {eval_trial_batch.time.values}")
# training_trial_batch = select_time.load()

tok = datetime.now()
# training_trial_batch = training_trial_batch.rename({'time': 'datetime'})


# with open("/Datastorage/saptarishi.dhanuka_asp25/era5_data/dataset_source-era5_date-2022-01-01_res-1.0_levels-13_steps-40.nc", 'rb') as f:
#     print("Loading Dataset")
#     tik = datetime.now()
#     training_trial_batch = xr.load_dataset(f).compute()
#     # training_trial_batch = training_trial_batch.isel(time=slice(0, 12))
#     # training_trial_batch = training_trial_batch.rename({'time': 'datetime'})
#     print("Training batch time")
#     print(training_trial_batch.datetime)
#     tok = datetime.now()
#     print(f"Dataset loaded in {tok - tik}")


with open(filename, 'rb') as f:
    ckpt = checkpoint.load(f, graphcast.CheckPoint)

params = ckpt.params


with open('/Datastorage/saptarishi.dhanuka_asp25/gc_norms/diffs_stddev_by_level.nc', 'rb') as f:
    diffs_stddev_by_level = xr.load_dataset(f).compute()
with open('/Datastorage/saptarishi.dhanuka_asp25/gc_norms/stddev_by_level.nc', 'rb') as f:
    stddev_by_level = xr.load_dataset(f).compute()
with open('/Datastorage/saptarishi.dhanuka_asp25/gc_norms/mean_by_level.nc', 'rb') as f:
    mean_by_level = xr.load_dataset(f).compute()


state = {}
model_config = ckpt.model_config
task_config = ckpt.task_config
setup_jax_functions.configs['model_config'] = model_config
setup_jax_functions.configs['task_config'] = task_config
setup_jax_functions.configs['state'] = state
setup_jax_functions.configs['params'] = params
setup_jax_functions.configs['stddev_by_level'] = stddev_by_level
setup_jax_functions.configs['diffs_stddev_by_level'] = diffs_stddev_by_level
setup_jax_functions.configs['mean_by_level'] = mean_by_level

setup_jax_functions.update_configs({
'params': ckpt.params,
'state': {},
'model_config': ckpt.model_config,
'task_config': ckpt.task_config,
'mean_by_level': mean_by_level,
'stddev_by_level': stddev_by_level,
'diffs_stddev_by_level': diffs_stddev_by_level
})

init_jitted = jax.jit(setup_jax_functions.with_configs(setup_jax_functions.run_forward.init))
loss_fn_jitted = setup_jax_functions.drop_state(setup_jax_functions.with_params(jax.jit(setup_jax_functions.with_configs(setup_jax_functions.loss_fn.apply))))
grads_fn_jitted = setup_jax_functions.with_params(jax.jit(setup_jax_functions.with_configs(setup_jax_functions.grads_fn)))
run_forward_jitted = setup_jax_functions.drop_state(setup_jax_functions.with_params(jax.jit(setup_jax_functions.with_configs(
setup_jax_functions.run_forward.apply))))

def run_model(params, state, inputs, targets_template, forcings):
    predictions = run_forward_jitted(
    rng=jax.random.PRNGKey(0),
    inputs=inputs,
    targets_template=targets_template,
    forcings=forcings,
    params=params,
    state=state
)
    return predictions

Coordinates after reassigning: Coordinates:
  * level     (level) int64 104B 50 100 150 200 250 300 ... 600 700 850 925 1000
  * lat       (lat) float64 1kB 90.0 89.0 88.0 87.0 ... -87.0 -88.0 -89.0 -90.0
  * lon       (lon) float64 3kB 0.0 1.0 2.0 3.0 4.0 ... 356.0 357.0 358.0 359.0
    datetime  (batch, time) datetime64[ns] 992B 2016-01-01 ... 2016-01-31T18:...
  * time      (time) timedelta64[ns] 992B 0 days 00:00:00 ... 30 days 18:00:00

Eval batch first time: [ 151200000000000  172800000000000  194400000000000  216000000000000
  237600000000000  259200000000000  280800000000000  302400000000000
  324000000000000  345600000000000  367200000000000  388800000000000
  410400000000000  432000000000000  453600000000000  475200000000000
  496800000000000  518400000000000  540000000000000  561600000000000
  583200000000000  604800000000000  626400000000000  648000000000000
  669600000000000  691200000000000  712800000000000  734400000000000
  756000000000000  777600000000000  799200000000

In [20]:
eval_trial_batch

<xarray.Dataset> Size: 3GB
Dimensions:                  (batch: 1, time: 64, lat: 181, lon: 360, level: 13)
Coordinates:
  * level                    (level) int64 104B 50 100 150 200 ... 850 925 1000
  * lat                      (lat) float64 1kB 90.0 89.0 88.0 ... -89.0 -90.0
  * lon                      (lon) float64 3kB 0.0 1.0 2.0 ... 357.0 358.0 359.0
    datetime                 (batch, time) datetime64[ns] 512B 2016-01-02T18:...
  * time                     (time) timedelta64[ns] 512B 1 days 18:00:00 ... ...
Dimensions without coordinates: batch
Data variables: (12/13)
    10m_u_component_of_wind  (batch, time, lat, lon) float64 33MB -0.7007 ......
    10m_v_component_of_wind  (batch, time, lat, lon) float64 33MB 0.06962 ......
    2m_temperature           (batch, time, lat, lon) float64 33MB 246.1 ... 2...
    geopotential             (batch, time, level, lat, lon) float64 434MB 1.8...
    geopotential_at_surface  (batch, lat, lon) float64 521kB -0.07617 ... 2.7...
    land_sea_mask            (batch, lat, lon) float64 521kB 0.0 0.0 ... 1.0 1.0
    ...                       ...
    specific_humidity        (batch, time, level, lat, lon) float64 434MB 2.8...
    temperature              (batch, time, level, lat, lon) float64 434MB 190...
    total_precipitation_6hr  (batch, time, lat, lon) float64 33MB 8.121e-07 ....
    u_component_of_wind      (batch, time, level, lat, lon) float64 434MB 0.0...
    v_component_of_wind      (batch, time, level, lat, lon) float64 434MB 0.0...
    vertical_velocity        (batch, time, level, lat, lon) float64 434MB 0.0...

In [8]:
time_len = len(eval_trial_batch.time.values)
time_len

64

In [9]:
eval_sim_data = eval_trial_batch.isel(time=slice(0, time_len-1))
eval_sim_data

<xarray.Dataset> Size: 3GB
Dimensions:                  (batch: 1, time: 63, lat: 181, lon: 360, level: 13)
Coordinates:
  * level                    (level) int64 104B 50 100 150 200 ... 850 925 1000
  * lat                      (lat) float64 1kB 90.0 89.0 88.0 ... -89.0 -90.0
  * lon                      (lon) float64 3kB 0.0 1.0 2.0 ... 357.0 358.0 359.0
    datetime                 (batch, time) datetime64[ns] 504B 2016-01-02T18:...
  * time                     (time) timedelta64[ns] 504B 1 days 18:00:00 ... ...
Dimensions without coordinates: batch
Data variables: (12/13)
    10m_u_component_of_wind  (batch, time, lat, lon) float64 33MB -0.7007 ......
    10m_v_component_of_wind  (batch, time, lat, lon) float64 33MB 0.06962 ......
    2m_temperature           (batch, time, lat, lon) float64 33MB 246.1 ... 2...
    geopotential             (batch, time, level, lat, lon) float64 427MB 1.8...
    geopotential_at_surface  (batch, lat, lon) float64 521kB -0.07617 ... 2.7...
    land_sea_mask            (batch, lat, lon) float64 521kB 0.0 0.0 ... 1.0 1.0
    ...                       ...
    specific_humidity        (batch, time, level, lat, lon) float64 427MB 2.8...
    temperature              (batch, time, level, lat, lon) float64 427MB 190...
    total_precipitation_6hr  (batch, time, lat, lon) float64 33MB 8.121e-07 ....
    u_component_of_wind      (batch, time, level, lat, lon) float64 427MB 0.0...
    v_component_of_wind      (batch, time, level, lat, lon) float64 427MB 0.0...
    vertical_velocity        (batch, time, level, lat, lon) float64 427MB 0.0...

In [10]:
new_params_path = "/Datastorage/saptarishi.dhanuka_asp25/gc_weights/graphcast_1_13_orig.npz"

f = open(new_params_path, "rb")
new_ckpt = checkpoint.load(f, graphcast.CheckPoint)
new_params = new_ckpt.params
f.close()

In [ ]:
jax.config.update("jax_enable_x64", True)

eval_sim_data = eval_trial_batch.isel(time=slice(0, time_len-1))
eval_inputs, eval_targets, eval_forcings = data_utils.extract_inputs_targets_forcings(
eval_sim_data, target_lead_times=slice("6h", "168h"),
**dataclasses.asdict(task_config))

assert eval_trial_batch.sizes["time"] >= 3

print(f"Eval batch time dimensions: {eval_trial_batch.sizes['time']}")

task_config_dict =  dataclasses.asdict(task_config)

print("Eval Inputs:   ", eval_inputs.dims.mapping)
print("Eval Targets:  ", eval_targets.dims.mapping)
print("Eval Forcings: ", eval_forcings.dims.mapping)

targets_template = eval_targets * np.nan

print("Running old params")
predictions_old = run_model(params, state, eval_inputs, targets_template, eval_forcings)

print("Predictions Old: ", predictions_old.dims.mapping)

print("Running new params")

predictions_finetuned = run_model(new_params, state, eval_inputs, targets_template, eval_forcings)
print("Predictions Finetuned: ", predictions_finetuned.dims.mapping)

predictions_old.to_netcdf(f"/Datastorage/saptarishi.dhanuka_asp25/prediction_nc_files/predictions_old_india_jan2016_0.nc")
predictions_finetuned.to_netcdf(f"/Datastorage/saptarishi.dhanuka_asp25/prediction_nc_files/predictions_finetuned_india2015_full_jan2016_0.nc")

Eval batch time dimensions: 64
Eval Inputs:    {'batch': 1, 'time': 2, 'lat': 181, 'lon': 360, 'level': 13}
Eval Targets:   {'batch': 1, 'time': 28, 'lat': 181, 'lon': 360, 'level': 13}
Eval Forcings:  {'batch': 1, 'time': 28, 'lat': 181, 'lon': 360}
Running old params
Predictions Old:  {'time': 28, 'batch': 1, 'lat': 181, 'lon': 360, 'level': 13}
Running new params
Predictions Finetuned:  {'time': 28, 'batch': 1, 'lat': 181, 'lon': 360, 'level': 13}


In [ ]:

temp_old = predictions_old['2m_temperature']
temp_ft  = predictions_finetuned['2m_temperature']

# align truth to forecast times
truth = eval_targets['2m_temperature'].sel(time=temp_old.time)


<xarray.DataArray 'time' (time: 28)> Size: 224B
array([ 21600000000000,  43200000000000,  64800000000000,  86400000000000,
       108000000000000, 129600000000000, 151200000000000, 172800000000000,
       194400000000000, 216000000000000, 237600000000000, 259200000000000,
       280800000000000, 302400000000000, 324000000000000, 345600000000000,
       367200000000000, 388800000000000, 410400000000000, 432000000000000,
       453600000000000, 475200000000000, 496800000000000, 518400000000000,
       540000000000000, 561600000000000, 583200000000000, 604800000000000],
      dtype='timedelta64[ns]')
Coordinates:
  * time     (time) timedelta64[ns] 224B 0 days 06:00:00 ... 7 days 00:00:00

In [30]:

temp_old = predictions_old['2m_temperature']
temp_ft  = predictions_finetuned['2m_temperature']

# align truth to forecast times
truth = eval_targets['2m_temperature'].sel(time=temp_old.time)


from pathlib import Path
PLOTS_BASE       = Path("./plots_2m_temperature_test")


import cartopy.crs as ccrs


def get_time_steps(ds, max_steps=7):
    """Return list of integer indices up to max_steps or dataset length."""
    return list(range(min(max_steps, ds.sizes.get('time', 0))))

LAT_MIN, LAT_MAX = 6, 38
LON_MIN, LON_MAX = 68, 98
START_DATE       = pd.to_datetime("2016-01-02T18:00:00.000000000")
for i in range(1):

    init_time = START_DATE + pd.Timedelta(days=i)
    print(init_time)
    init_str  = init_time.strftime("%Y%m%dT%H%M%S")
    predictions_old = xr.open_dataset(f"/Datastorage/saptarishi.dhanuka_asp25/prediction_nc_files/predictions_old_india_jan2016_{i}.nc")
    predictions_finetuned = xr.open_dataset(f"/Datastorage/saptarishi.dhanuka_asp25/prediction_nc_files/predictions_finetuned_india2015_full_jan2016_{i}.nc")



    temp_old = predictions_old['2m_temperature']
    temp_ft  = predictions_finetuned['2m_temperature']

    # align truth to forecast times
    truth = eval_targets['2m_temperature'].sel(time=temp_old.time)

    # prepare output folders
    comp_dir = PLOTS_BASE / init_str / "comparison"
    diff_dir = PLOTS_BASE / init_str / "difference_vs_truth"
    comp_dir.mkdir(parents=True, exist_ok=True)
    diff_dir.mkdir(parents=True, exist_ok=True)

    # determine which lead steps to plot
    time_steps = get_time_steps(predictions_old)




    mse_old = []
    mse_ft  = []
    for t in time_steps:
        err2_old = (temp_old.isel(time=t) - truth.isel(time=t))**2
        err2_ft  = (temp_ft.isel(time=t)  - truth.isel(time=t))**2
        mse_old.append(err2_old.mean(dim=('lat','lon')).item())
        mse_ft.append(err2_ft.mean(dim=('lat','lon')).item())
    
    # Plot MSE time series
    line_dir = PLOTS_BASE / init_str / "mse_line"
    line_dir.mkdir(parents=True, exist_ok=True)
    
    plt.figure(figsize=(10, 6))
    plt.plot(time_steps, mse_old, marker='o', label='Base Model')
    plt.plot(time_steps, mse_ft,  marker='s', label='Fine-tuned Model')
    plt.xlabel('Lead Time (index)')
    plt.ylabel('Mean Squared Error (2m_temperature)')
    plt.title(f"MSE over Lead Time – init {init_str}")
    plt.legend()
    plt.grid(True)
    plt.savefig(line_dir / f"mse_line_{init_str}.png")
    plt.close()



2016-01-02 18:00:00


/tmp/ipykernel_1520734/578901343.py:27: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  predictions_old = xr.open_dataset(f"/Datastorage/saptarishi.dhanuka_asp25/prediction_nc_files/predictions_old_india_jan2016_{i}.nc")
/tmp/ipykernel_1520734/578901343.py:28: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  predictions_finetuned = xr.open_dataset(f"/Datastorage/saptarishi.dhanuka_asp25/prediction_nc_files/predictions_finetuned_india2015_full_jan2016_{i}.nc")


In [32]:
# i should be max till time_len - 28 so that can roll out 7 day forecast and also verify it
for i in tqdm(range(10)):
    eval_sim_data = eval_trial_batch.isel(time=slice(i*4, time_len-1))
    eval_inputs, eval_targets, eval_forcings = data_utils.extract_inputs_targets_forcings(
    eval_sim_data, target_lead_times=slice("6h", "168h"),
    **dataclasses.asdict(task_config))

    assert eval_trial_batch.sizes["time"] >= 3

    print(f"Eval batch time dimensions: {eval_trial_batch.sizes['time']}")

    task_config_dict =  dataclasses.asdict(task_config)

    print("Eval Inputs:   ", eval_inputs.dims.mapping)
    print("Eval Targets:  ", eval_targets.dims.mapping)
    print("Eval Forcings: ", eval_forcings.dims.mapping)

    targets_template = eval_targets * np.nan

    print("Running old params")
    predictions_old = run_model(params, state, eval_inputs, targets_template, eval_forcings)

    print("Predictions Old: ", predictions_old.dims.mapping)

    print("Running new params")

    predictions_finetuned = run_model(new_params, state, eval_inputs, targets_template, eval_forcings)
    print("Predictions Finetuned: ", predictions_finetuned.dims.mapping)

    predictions_old.to_netcdf(f"/Datastorage/saptarishi.dhanuka_asp25/prediction_nc_files/predictions_old_india_jan2016_{i}.nc")
    predictions_finetuned.to_netcdf(f"/Datastorage/saptarishi.dhanuka_asp25/prediction_nc_files/predictions_finetuned_india2015_full_jan2016_{i}.nc")

  0%|          | 0/10 [00:00<?, ?it/s]

Eval batch time dimensions: 64
Eval Inputs:    {'batch': 1, 'time': 2, 'lat': 181, 'lon': 360, 'level': 13}
Eval Targets:   {'batch': 1, 'time': 28, 'lat': 181, 'lon': 360, 'level': 13}
Eval Forcings:  {'batch': 1, 'time': 28, 'lat': 181, 'lon': 360}
Running old params
Predictions Old:  {'time': 28, 'batch': 1, 'lat': 181, 'lon': 360, 'level': 13}
Running new params
Predictions Finetuned:  {'time': 28, 'batch': 1, 'lat': 181, 'lon': 360, 'level': 13}


 10%|█         | 1/10 [04:45<42:45, 285.06s/it]

Eval batch time dimensions: 64
Eval Inputs:    {'batch': 1, 'time': 2, 'lat': 181, 'lon': 360, 'level': 13}
Eval Targets:   {'batch': 1, 'time': 28, 'lat': 181, 'lon': 360, 'level': 13}
Eval Forcings:  {'batch': 1, 'time': 28, 'lat': 181, 'lon': 360}
Running old params
Predictions Old:  {'time': 28, 'batch': 1, 'lat': 181, 'lon': 360, 'level': 13}
Running new params
Predictions Finetuned:  {'time': 28, 'batch': 1, 'lat': 181, 'lon': 360, 'level': 13}


 20%|██        | 2/10 [10:20<41:56, 314.51s/it]

...

Eval batch time dimensions: 64
Eval Inputs:    {'batch': 1, 'time': 2, 'lat': 181, 'lon': 360, 'level': 13}
Eval Targets:   {'batch': 1, 'time': 28, 'lat': 181, 'lon': 360, 'level': 13}
Eval Forcings:  {'batch': 1, 'time': 28, 'lat': 181, 'lon': 360}
Running old params
Predictions Old:  {'time': 28, 'batch': 1, 'lat': 181, 'lon': 360, 'level': 13}
Running new params
Predictions Finetuned:  {'time': 28, 'batch': 1, 'lat': 181, 'lon': 360, 'level': 13}


 90%|█████████ | 9/10 [45:04<04:28, 268.10s/it]

Eval batch time dimensions: 64
Eval Inputs:    {'batch': 1, 'time': 0, 'lat': 181, 'lon': 360, 'level': 13}
Eval Targets:   {'batch': 1, 'time': 27, 'lat': 181, 'lon': 360, 'level': 13}
Eval Forcings:  {'batch': 1, 'time': 27, 'lat': 181, 'lon': 360}
Running old params


 90%|█████████ | 9/10 [45:09<05:01, 301.10s/it]


ValueError: 'grid2mesh_gnn/~_networks_builder/encoder_nodes_grid_nodes_mlp/~/linear_0/w' with retrieved shape (186, 512) does not match shape=[10, 512] dtype=dtype(bfloat16)

In [28]:
predictions_old = xr.open_dataset(f"/Datastorage/saptarishi.dhanuka_asp25/prediction_nc_files/predictions_old_india_jan2016{i}.nc")
predictions_finetuned = xr.open_dataset(f"/Datastorage/saptarishi.dhanuka_asp25/prediction_nc_files/predictions_finetuned_india2015_full_jan2016{i}.nc")

/tmp/ipykernel_1507129/2243525479.py:1: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  predictions_old = xr.open_dataset(f"/Datastorage/saptarishi.dhanuka_asp25/prediction_nc_files/predictions_old_india_jan2016{i}.nc")
/tmp/ipykernel_1507129/2243525479.py:2: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  predictions_finetuned = xr.open_dataset(f"/Datastorage/saptarishi.dhanuka_asp25/prediction_nc_files/predictions_finetuned_india2015_full_jan2016{i}.nc")


In [ ]:

temp_old = predictions_old['2m_temperature']
temp_ft  = predictions_finetuned['2m_temperature']

# align truth to forecast times
truth = eval_targets['2m_temperature'].sel(time=temp_old.time)


from pathlib import Path
PLOTS_BASE       = Path("./plots_2m_temperature")


import cartopy.crs as ccrs


def get_time_steps(ds, max_steps=7):
    """Return list of integer indices up to max_steps or dataset length."""
    return list(range(min(max_steps, ds.sizes.get('time', 0))))

LAT_MIN, LAT_MAX = 6, 38
LON_MIN, LON_MAX = 68, 98
START_DATE       = pd.to_datetime("2016-01-02T18:00:00.000000000")
for i in range(2):

    init_time = START_DATE + pd.Timedelta(days=i)
    print(init_time)
    init_str  = init_time.strftime("%Y%m%dT%H%M%S")
    predictions_old = xr.open_dataset(f"/Datastorage/saptarishi.dhanuka_asp25/prediction_nc_files/predictions_old_india_jan2016{i}.nc")
    predictions_finetuned = xr.open_dataset(f"/Datastorage/saptarishi.dhanuka_asp25/prediction_nc_files/predictions_finetuned_india2015_full_jan2016{i}.nc")



    temp_old = predictions_old['2m_temperature']
    temp_ft  = predictions_finetuned['2m_temperature']

    # align truth to forecast times
    truth = eval_targets['2m_temperature'].sel(time=temp_old.time)

    # prepare output folders
    comp_dir = PLOTS_BASE / init_str / "comparison"
    diff_dir = PLOTS_BASE / init_str / "difference_vs_truth"
    comp_dir.mkdir(parents=True, exist_ok=True)
    diff_dir.mkdir(parents=True, exist_ok=True)

    # determine which lead steps to plot
    time_steps = get_time_steps(predictions_old)




    mse_old = []
    mse_ft  = []
    for t in time_steps:
        err2_old = (temp_old.isel(time=t) - truth.isel(time=t))**2
        err2_ft  = (temp_ft.isel(time=t)  - truth.isel(time=t))**2
        mse_old.append(err2_old.mean(dim=('lat','lon')).item())
        mse_ft.append(err2_ft.mean(dim=('lat','lon')).item())
    
    # Plot MSE time series
    line_dir = PLOTS_BASE / init_str / "mse_line"
    line_dir.mkdir(parents=True, exist_ok=True)
    
    plt.figure(figsize=(10, 6))
    plt.plot(time_steps, mse_old, marker='o', label='Base Model')
    plt.plot(time_steps, mse_ft,  marker='s', label='Fine-tuned Model')
    plt.xlabel('Lead Time (index)')
    plt.ylabel('Mean Squared Error (2m_temperature)')
    plt.title(f"MSE over Lead Time – init {init_str}")
    plt.legend()
    plt.grid(True)
    plt.savefig(line_dir / f"mse_line_{init_str}.png")
    plt.close()



<xarray.DataArray '2m_temperature' (batch: 1, time: 28, lat: 181, lon: 360)> Size: 15MB
array([[[[253.10223389, 253.10223389, 253.10223389, ..., 253.10223389,
          253.10223389, 253.10223389],
         [252.40698242, 252.38305664, 252.35772705, ..., 252.50408936,
          252.47172546, 252.43934631],
         [252.07624817, 252.04528809, 252.014328  , ..., 252.20571899,
          252.15083313, 252.11424255],
         ...,
         [245.31938171, 245.27998352, 245.24057007, ..., 245.36019897,
          245.34190369, 245.3306427 ],
         [245.40805054, 245.40242004, 245.39538574, ..., 245.43760681,
          245.42774963, 245.41790771],
         [244.36236572, 244.36236572, 244.36236572, ..., 244.36236572,
          244.36236572, 244.36236572]],

        [[250.78990173, 250.78990173, 250.78990173, ..., 250.78990173,
          250.78990173, 250.78990173],
         [250.6139679 , 250.59286499, 250.56893921, ..., 250.69137573,
          250.66604614, 250.63931274],
         [250.65618896, 250.62522888, 250.5942688 , ..., 250.72093201,
          250.69419861, 250.67590332],
...
         [242.6212616 , 242.53894043, 242.456604  , ..., 242.78166199,
          242.7305603 , 242.67520142],
         [242.95909119, 242.94915771, 242.93922424, ..., 243.0002594 ,
          242.98606873, 242.97329712],
         [243.11665344, 243.11665344, 243.11665344, ..., 243.11665344,
          243.11665344, 243.11665344]],

        [[252.9564209 , 252.9564209 , 252.9564209 , ..., 252.9564209 ,
          252.9564209 , 252.9564209 ],
         [251.62780762, 251.71154785, 251.79672241, ..., 251.44326782,
          251.50288391, 251.56533813],
         [251.03019714, 251.13949585, 251.24737549, ..., 250.69236755,
          250.79882812, 250.91522217],
         ...,
         [242.29620361, 242.25361633, 242.20960999, ..., 242.38847351,
          242.3600769 , 242.32743835],
         [242.89379883, 242.88386536, 242.87251282, ..., 242.93496704,
          242.92077637, 242.9079895 ],
         [242.68655396, 242.68655396, 242.68655396, ..., 242.68655396,
          242.68655396, 242.68655396]]]])
Coordinates:
  * lat      (lat) float64 1kB 90.0 89.0 88.0 87.0 ... -87.0 -88.0 -89.0 -90.0
  * lon      (lon) float64 3kB 0.0 1.0 2.0 3.0 4.0 ... 356.0 357.0 358.0 359.0
  * time     (time) timedelta64[ns] 224B 0 days 06:00:00 ... 7 days 00:00:00
Dimensions without coordinates: batch
Attributes:
    long_name:   2 metre temperature
    short_name:  t2m
    units:       K

In [38]:
%reset -f

In [37]:
from pathlib import Path
PLOTS_BASE       = Path("./plots_2m_temperature")


import cartopy.crs as ccrs


def get_time_steps(ds, max_steps=7):
    """Return list of integer indices up to max_steps or dataset length."""
    return list(range(min(max_steps, ds.sizes.get('time', 0))))

LAT_MIN, LAT_MAX = 6, 38
LON_MIN, LON_MAX = 68, 98
START_DATE       = pd.to_datetime("2016-01-02T18:00:00.000000000")
for i in tqdm(range(8)):

    init_time = START_DATE + pd.Timedelta(days=i)
    init_str  = init_time.strftime("%Y%m%dT%H%M%S")
    predictions_old = xr.open_dataset(f"/Datastorage/saptarishi.dhanuka_asp25/prediction_nc_files/predictions_old_india_jan2016_{i}.nc")
    predictions_finetuned = xr.open_dataset(f"/Datastorage/saptarishi.dhanuka_asp25/prediction_nc_files/predictions_finetuned_india2015_full_jan2016_{i}.nc")



    temp_old = predictions_old['2m_temperature']
    temp_ft  = predictions_finetuned['2m_temperature']

    # drop the first time index data
    temp_old = temp_old.isel(time=slice(1, None))

    print(temp_old.time)
    print()

    print(eval_targets['2m_temperature'].time)
    # align truth to forecast times
    truth = eval_targets['2m_temperature'].sel(time=temp_old.time)

    # prepare output folders
    comp_dir = PLOTS_BASE / init_str / "comparison"
    diff_dir = PLOTS_BASE / init_str / "difference_vs_truth"
    comp_dir.mkdir(parents=True, exist_ok=True)
    diff_dir.mkdir(parents=True, exist_ok=True)

    # determine which lead steps to plot
    time_steps = get_time_steps(predictions_old)




    mse_old = []
    mse_ft  = []
    for t in time_steps:
        err2_old = (temp_old.isel(time=t) - truth.isel(time=t))**2
        err2_ft  = (temp_ft.isel(time=t)  - truth.isel(time=t))**2
        mse_old.append(err2_old.mean(dim=('lat','lon')).item())
        mse_ft.append(err2_ft.mean(dim=('lat','lon')).item())
    
    # Plot MSE time series
    line_dir = PLOTS_BASE / init_str / "mse_line"
    line_dir.mkdir(parents=True, exist_ok=True)
    
    plt.figure(figsize=(10, 6))
    plt.plot(time_steps, mse_old, marker='o', label='Base Model')
    plt.plot(time_steps, mse_ft,  marker='s', label='Fine-tuned Model')
    plt.xlabel('Lead Time (index)')
    plt.ylabel('Mean Squared Error (2m_temperature)')
    plt.title(f"MSE over Lead Time – init {init_str}")
    plt.legend()
    plt.grid(True)
    plt.savefig(line_dir / f"mse_line_{init_str}.png")
    plt.close()




    # for t in tqdm(time_steps, desc=f"Init {init_str}"):
    #     # 1) three-panel comparison
    #     fig, axs = plt.subplots(1, 3, figsize=(18, 6),
    #                             subplot_kw={'projection': ccrs.PlateCarree()})
    #     panels = [
    #         (temp_old.isel(time=t),           f"Old @ t={t}"),
    #         (temp_ft .isel(time=t),           f"Finetuned @ t={t}"),
    #         (temp_ft.isel(time=t) - temp_old.isel(time=t), f"Diff @ t={t}")
    #     ]
    #     for ax, (data, title) in zip(axs, panels):
    #         data.squeeze().plot.pcolormesh(
    #             ax=ax,
    #             transform=ccrs.PlateCarree(),
    #             cmap='bwr' if "Diff" in title else 'coolwarm',
    #             cbar_kwargs={'label':'2m_temperature (K)'}
    #         )
    #         ax.set_extent([LON_MIN, LON_MAX, LAT_MIN, LAT_MAX])
    #         ax.coastlines()
    #         ax.set_title(title)
    #     fig.tight_layout()
    #     fig.savefig(comp_dir / f"comparison_t{t}.png")
    #     plt.close(fig)

    #     # 2) finetuned minus truth
    #     fig, ax = plt.subplots(1,1,figsize=(8,6),
    #                            subplot_kw={'projection': ccrs.PlateCarree()})
    #     diff_truth = temp_ft.isel(time=t) - truth.isel(time=t)
    #     diff_truth.squeeze().plot.pcolormesh(
    #         ax=ax,
    #         transform=ccrs.PlateCarree(),
    #         cmap='bwr',
    #         cbar_kwargs={'label':'Δ Temperature (K)'}
    #     )
    #     ax.set_extent([LON_MIN, LON_MAX, LAT_MIN, LAT_MAX])
    #     ax.coastlines()
    #     ax.set_title(f"Finetuned − Truth @ t={t}")
    #     fig.tight_layout()
    #     fig.savefig(diff_dir / f"diff_truth_t{t}.png")
    #     plt.close(fig)

  0%|          | 0/8 [00:00<?, ?it/s]/tmp/ipykernel_1520734/2402647451.py:19: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  predictions_old = xr.open_dataset(f"/Datastorage/saptarishi.dhanuka_asp25/prediction_nc_files/predictions_old_india_jan2016_{i}.nc")
/tmp/ipykernel_1520734/2402647451.py:20: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  predictions_finetuned = xr.open_dataset(f"/Datastorage/saptarishi.dhanuka_asp25/prediction_nc_files/predictions_finetuned_india2015_full_jan2016_{i}.nc")
 12%|█▎        | 1/8 [00:00<00:00,  9.27it/s]/tmp/ipykernel_1520734/2402647451.py:19: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To sil

<xarray.DataArray 'time' (time: 27)> Size: 216B
array([ 43200000000000,  64800000000000,  86400000000000, 108000000000000,
       129600000000000, 151200000000000, 172800000000000, 194400000000000,
       216000000000000, 237600000000000, 259200000000000, 280800000000000,
       302400000000000, 324000000000000, 345600000000000, 367200000000000,
       388800000000000, 410400000000000, 432000000000000, 453600000000000,
       475200000000000, 496800000000000, 518400000000000, 540000000000000,
       561600000000000, 583200000000000, 604800000000000],
      dtype='timedelta64[ns]')
Coordinates:
  * time     (time) timedelta64[ns] 216B 0 days 12:00:00 ... 7 days 00:00:00

<xarray.DataArray 'time' (time: 27)> Size: 216B
array([ 43200000000000,  64800000000000,  86400000000000, 108000000000000,
       129600000000000, 151200000000000, 172800000000000, 194400000000000,
       216000000000000, 237600000000000, 259200000000000, 280800000000000,
       302400000000000, 324000000000000, 3456000

/tmp/ipykernel_1520734/2402647451.py:19: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  predictions_old = xr.open_dataset(f"/Datastorage/saptarishi.dhanuka_asp25/prediction_nc_files/predictions_old_india_jan2016_{i}.nc")
/tmp/ipykernel_1520734/2402647451.py:20: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  predictions_finetuned = xr.open_dataset(f"/Datastorage/saptarishi.dhanuka_asp25/prediction_nc_files/predictions_finetuned_india2015_full_jan2016_{i}.nc")
/tmp/ipykernel_1520734/2402647451.py:19: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' in

<xarray.DataArray 'time' (time: 27)> Size: 216B
array([ 43200000000000,  64800000000000,  86400000000000, 108000000000000,
       129600000000000, 151200000000000, 172800000000000, 194400000000000,
       216000000000000, 237600000000000, 259200000000000, 280800000000000,
       302400000000000, 324000000000000, 345600000000000, 367200000000000,
       388800000000000, 410400000000000, 432000000000000, 453600000000000,
       475200000000000, 496800000000000, 518400000000000, 540000000000000,
       561600000000000, 583200000000000, 604800000000000],
      dtype='timedelta64[ns]')
Coordinates:
  * time     (time) timedelta64[ns] 216B 0 days 12:00:00 ... 7 days 00:00:00

<xarray.DataArray 'time' (time: 27)> Size: 216B
array([ 43200000000000,  64800000000000,  86400000000000, 108000000000000,
       129600000000000, 151200000000000, 172800000000000, 194400000000000,
       216000000000000, 237600000000000, 259200000000000, 280800000000000,
       302400000000000, 324000000000000, 3456000

/tmp/ipykernel_1520734/2402647451.py:19: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  predictions_old = xr.open_dataset(f"/Datastorage/saptarishi.dhanuka_asp25/prediction_nc_files/predictions_old_india_jan2016_{i}.nc")
/tmp/ipykernel_1520734/2402647451.py:20: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  predictions_finetuned = xr.open_dataset(f"/Datastorage/saptarishi.dhanuka_asp25/prediction_nc_files/predictions_finetuned_india2015_full_jan2016_{i}.nc")
 75%|███████▌  | 6/8 [00:00<00:00, 10.10it/s]/tmp/ipykernel_1520734/2402647451.py:19: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedel

<xarray.DataArray 'time' (time: 27)> Size: 216B
array([ 43200000000000,  64800000000000,  86400000000000, 108000000000000,
       129600000000000, 151200000000000, 172800000000000, 194400000000000,
       216000000000000, 237600000000000, 259200000000000, 280800000000000,
       302400000000000, 324000000000000, 345600000000000, 367200000000000,
       388800000000000, 410400000000000, 432000000000000, 453600000000000,
       475200000000000, 496800000000000, 518400000000000, 540000000000000,
       561600000000000, 583200000000000, 604800000000000],
      dtype='timedelta64[ns]')
Coordinates:
  * time     (time) timedelta64[ns] 216B 0 days 12:00:00 ... 7 days 00:00:00

<xarray.DataArray 'time' (time: 27)> Size: 216B
array([ 43200000000000,  64800000000000,  86400000000000, 108000000000000,
       129600000000000, 151200000000000, 172800000000000, 194400000000000,
       216000000000000, 237600000000000, 259200000000000, 280800000000000,
       302400000000000, 324000000000000, 3456000

100%|██████████| 8/8 [00:00<00:00, 10.06it/s]
